Google Drive Mount

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check that Colab can see the files

In [16]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/Project2_SolarPV")

print("Folder exists:", folder.exists())

for filename in [
    "Plant_1_Generation_Data.csv",
    "Plant_1_Weather_Sensor_Data.csv",
    "Plant_2_Generation_Data.csv",
    "Plant_2_Weather_Sensor_Data.csv",
]:
    print(filename, "FOUND" if (folder / filename).exists() else "MISSING")

Folder exists: True
Plant_1_Generation_Data.csv FOUND
Plant_1_Weather_Sensor_Data.csv FOUND
Plant_2_Generation_Data.csv FOUND
Plant_2_Weather_Sensor_Data.csv FOUND


Import libraries and read the CSV files

In [17]:
import pandas as pd
import numpy as np

p1_gen_raw = pd.read_csv(folder / "Plant_1_Generation_Data.csv")
p1_wth_raw = pd.read_csv(folder / "Plant_1_Weather_Sensor_Data.csv")
p2_gen_raw = pd.read_csv(folder / "Plant_2_Generation_Data.csv")
p2_wth_raw = pd.read_csv(folder / "Plant_2_Weather_Sensor_Data.csv")

datasets = {
    "Plant 1 Generation": p1_gen_raw,
    "Plant 1 Weather": p1_wth_raw,
    "Plant 2 Generation": p2_gen_raw,
    "Plant 2 Weather": p2_wth_raw,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


Plant 1 Generation
Shape: (68778, 7)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.0,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.0,6987759.0



Plant 1 Weather
Shape: (3182, 6)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4135001,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,4135001,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,4135001,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0



Plant 2 Generation
Shape: (67698, 7)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,2020-05-15 00:00:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,9425.000000,2.429011e+06
1,2020-05-15 00:00:00,4136001,81aHJ1q11NBPMrL,0.0,0.0,0.000000,1.215279e+09
2,2020-05-15 00:00:00,4136001,9kRcWv60rDACzjR,0.0,0.0,3075.333333,2.247720e+09



Plant 2 Weather
Shape: (3259, 6)
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4136001,iq8k7ZNt4Mwm3w0,27.004764,25.060789,0.0
1,2020-05-15 00:15:00,4136001,iq8k7ZNt4Mwm3w0,26.880811,24.421869,0.0
2,2020-05-15 00:30:00,4136001,iq8k7ZNt4Mwm3w0,26.682055,24.427290,0.0


Check the timestamp formats

In [18]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("First timestamps:", df["DATE_TIME"].head(3).tolist())
    print("Timestamp string lengths:")
    display(df["DATE_TIME"].astype(str).str.len().value_counts())


Plant 1 Generation
First timestamps: ['15-05-2020 00:00', '15-05-2020 00:00', '15-05-2020 00:00']
Timestamp string lengths:


,count
DATE_TIME,
16,68778



Plant 1 Weather
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:15:00', '2020-05-15 00:30:00']
Timestamp string lengths:


,count
DATE_TIME,
19,3182



Plant 2 Generation
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:00:00', '2020-05-15 00:00:00']
Timestamp string lengths:


,count
DATE_TIME,
19,67698



Plant 2 Weather
First timestamps: ['2020-05-15 00:00:00', '2020-05-15 00:15:00', '2020-05-15 00:30:00']
Timestamp string lengths:


,count
DATE_TIME,
19,3259


Convert timestamps into dates

In [19]:
date_formats = {
    "Plant 1 Generation": "%d-%m-%Y %H:%M",
    "Plant 1 Weather": "%Y-%m-%d %H:%M:%S",
    "Plant 2 Generation": "%Y-%m-%d %H:%M:%S",
    "Plant 2 Weather": "%Y-%m-%d %H:%M:%S",
}

parsed = {}

for name, df in datasets.items():
    df = df.copy()
    df["DATE_TIME"] = pd.to_datetime(
        df["DATE_TIME"],
        format=date_formats[name],
        errors="raise"
    )
    parsed[name] = df

p1_gen = parsed["Plant 1 Generation"]
p1_wth = parsed["Plant 1 Weather"]
p2_gen = parsed["Plant 2 Generation"]
p2_wth = parsed["Plant 2 Weather"]

print("All timestamps converted successfully.")

All timestamps converted successfully.


Check each file’s rows, date range, missing values, and power

In [20]:
def summarize_dataset(name, df, value_column=None):
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)
    print("Rows:", len(df))
    print("Date range:", df["DATE_TIME"].min(), "to", df["DATE_TIME"].max())

    if "SOURCE_KEY" in df.columns:
        print("Unique sources/inverters:", df["SOURCE_KEY"].nunique())

    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count"))

    if value_column:
        print(f"{value_column} summary:")
        display(df[value_column].describe().to_frame())
        print("Negative values:", (df[value_column] < 0).sum())

summarize_dataset("Plant 1 Generation", p1_gen, "AC_POWER")
summarize_dataset("Plant 1 Weather", p1_wth, "IRRADIATION")
summarize_dataset("Plant 2 Generation", p2_gen, "AC_POWER")
summarize_dataset("Plant 2 Weather", p2_wth, "IRRADIATION")


Plant 1 Generation
Rows: 68778
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 22
Missing values:


,missing_count
DATE_TIME,0
PLANT_ID,0
SOURCE_KEY,0
DC_POWER,0
AC_POWER,0
DAILY_YIELD,0
TOTAL_YIELD,0


AC_POWER summary:


,AC_POWER
count,68778.000000
mean,307.802752
std,394.396439
min,0.000000
25%,0.000000
50%,41.493750
75%,623.618750
max,1410.950000


Negative values: 0

Plant 1 Weather
Rows: 3182
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 1
Missing values:


...

Negative values: 0

Plant 2 Weather
Rows: 3259
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Unique sources/inverters: 1
Missing values:


,missing_count
DATE_TIME,0
PLANT_ID,0
SOURCE_KEY,0
AMBIENT_TEMPERATURE,0
MODULE_TEMPERATURE,0
IRRADIATION,0


IRRADIATION summary:


,IRRADIATION
count,3259.000000
mean,0.232737
std,0.312693
min,0.000000
25%,0.000000
50%,0.019040
75%,0.438717
max,1.098766


Negative values: 0


##Compare generation and weather timestamps
This checks how many timestamps are shared between generation and weather data.

---



In [21]:
def compare_timestamps(plant_name, generation_df, weather_df):
    generation_times = set(generation_df["DATE_TIME"].unique())
    weather_times = set(weather_df["DATE_TIME"].unique())

    common = generation_times & weather_times
    generation_only = generation_times - weather_times
    weather_only = weather_times - generation_times

    print(f"\n{plant_name}")
    print("Generation timestamps:", len(generation_times))
    print("Weather timestamps:", len(weather_times))
    print("Shared timestamps:", len(common))
    print("Generation timestamps without weather:", len(generation_only))
    print("Weather timestamps without generation:", len(weather_only))

compare_timestamps("Plant 1", p1_gen, p1_wth)
compare_timestamps("Plant 2", p2_gen, p2_wth)


Plant 1
Generation timestamps: 3158
Weather timestamps: 3182
Shared timestamps: 3157
Generation timestamps without weather: 1
Weather timestamps without generation: 25

Plant 2
Generation timestamps: 3259
Weather timestamps: 3259
Shared timestamps: 3259
Generation timestamps without weather: 0
Weather timestamps without generation: 0


Compare inverter daytime records

In [22]:
def inverter_daytime_summary(plant_name, generation_df):
    df = generation_df.copy()
    df["HOUR"] = df["DATE_TIME"].dt.hour

    daytime = df[df["HOUR"].between(6, 18)]

    summary = daytime.groupby("SOURCE_KEY")["AC_POWER"].agg(
        record_count="count",
        mean_power="mean",
        median_power="median",
        max_power="max",
        std_power="std",
    ).round(2)

    print(f"\n{plant_name}: daytime inverter power")
    display(summary)

    print("Number of inverters:", len(summary))
    print(
        "Daytime records per inverter, minimum to maximum:",
        summary["record_count"].min(),
        "to",
        summary["record_count"].max()
    )
    print("Inverters with zero maximum daytime power:",
          (summary["max_power"] == 0).sum())

    return summary

p1_inverter_summary = inverter_daytime_summary("Plant 1", p1_gen)
p2_inverter_summary = inverter_daytime_summary("Plant 2", p2_gen)


Plant 1: daytime inverter power


,record_count,mean_power,median_power,max_power,std_power
SOURCE_KEY,,,,,
1BY6WEcLGh8j5v7,1733,511.64,512.53,1300.17,357.84
1IF53ai7Xc0U56Y,1719,572.43,583.91,1394.51,382.84
3PZuoBAID5Wc2HD,1718,571.63,583.77,1399.22,383.55
7JYdWkrLSPkdwr4,1729,555.18,566.74,1384.87,375.15
McdE0feGgRqW7Ca,1723,569.40,581.83,1405.80,384.71
VHMLBKoKgIrUVDU,1729,568.80,582.59,1394.21,384.52
WRmjgnKYAwPKWDb,1718,560.03,569.45,1376.35,374.75
YxYtjZvoooNbGkE,1721,560.79,569.06,1410.53,377.21
ZnxXDlPa8U1GXgE,1726,566.42,580.12,1391.64,382.00


Number of inverters: 22
Daytime records per inverter, minimum to maximum: 1718 to 1734
Inverters with zero maximum daytime power: 0

Plant 2: daytime inverter power


,record_count,mean_power,median_power,max_power,std_power
SOURCE_KEY,,,,,
4UPUqMRk7TRMgml,1731,501.26,447.51,1384.35,410.99
81aHJ1q11NBPMrL,1764,417.03,336.76,1347.66,386.52
9kRcWv60rDACzjR,1764,442.07,379.00,1361.67,389.42
Et9kgGMDl729KT4,1731,340.04,241.61,1172.65,335.78
IQ2d7wF4YD8zU1Q,1275,515.68,466.36,1383.46,395.60
LYwnQax7tkwH5Cb,1764,354.89,261.52,1265.67,344.97
LlT2YUhhzqhg5Sw,1764,443.61,384.58,1338.35,387.78
Mx2yZCDsyf6DPfv,1731,514.34,464.66,1377.15,411.83
NgDl19wMapZy17u,1275,494.06,450.09,1370.96,386.10


Number of inverters: 22
Daytime records per inverter, minimum to maximum: 1275 to 1764
Inverters with zero maximum daytime power: 0


##Compare plant-level daytime power
This adds together the power from all inverters at each timestamp.

In [23]:
def plant_daytime_power(plant_name, generation_df):
    df = generation_df.copy()
    df["HOUR"] = df["DATE_TIME"].dt.hour

    daytime = df[df["HOUR"].between(6, 18)]

    plant_power = daytime.groupby("DATE_TIME")["AC_POWER"].sum()

    print(f"\n{plant_name}: plant-level daytime AC_POWER")
    display(plant_power.describe().to_frame("AC_POWER"))

    return plant_power

p1_daytime_power = plant_daytime_power("Plant 1", p1_gen)
p2_daytime_power = plant_daytime_power("Plant 2", p2_gen)


Plant 1: plant-level daytime AC_POWER


,AC_POWER
count,1736.000000
mean,12194.733695
std,8226.413234
min,0.000000
25%,4351.973214
50%,12555.564286
75%,18903.314732
max,29150.212499



Plant 2: plant-level daytime AC_POWER


,AC_POWER
count,1764.000000
mean,9259.647108
std,5865.926740
min,0.000000
25%,3622.434881
50%,10267.086190
75%,13904.742262
max,25979.760476


Combine Plant 1 generation and weather

In [24]:
gen_plant = (
    p1_gen.groupby("DATE_TIME", as_index=False)
    .agg(AC_POWER=("AC_POWER", "sum"))
    .sort_values("DATE_TIME")
)

wth_clean = p1_wth[
    [
        "DATE_TIME",
        "AMBIENT_TEMPERATURE",
        "MODULE_TEMPERATURE",
        "IRRADIATION",
    ]
].copy()

print("Duplicate weather timestamps:", wth_clean["DATE_TIME"].duplicated().sum())

if wth_clean["DATE_TIME"].duplicated().any():
    raise ValueError("There are duplicate weather timestamps. Check the weather file.")

merged = pd.merge(
    gen_plant,
    wth_clean,
    on="DATE_TIME",
    how="inner",
    validate="one_to_one"
).sort_values("DATE_TIME").reset_index(drop=True)

print("Generation timestamps before merge:", len(gen_plant))
print("Weather timestamps:", len(wth_clean))
print("Rows after merge:", len(merged))
print("Generation timestamps not included after merge:", len(gen_plant) - len(merged))
print("Columns:", merged.columns.tolist())
print("Missing values:")
display(merged.isna().sum().to_frame("missing_count"))
print("Date range:", merged["DATE_TIME"].min(), "to", merged["DATE_TIME"].max())
display(merged.head())

Duplicate weather timestamps: 0
Generation timestamps before merge: 3158
Weather timestamps: 3182
Rows after merge: 3157
Generation timestamps not included after merge: 1
Columns: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
Missing values:


,missing_count
DATE_TIME,0
AC_POWER,0
AMBIENT_TEMPERATURE,0
MODULE_TEMPERATURE,0
IRRADIATION,0


Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00


,DATE_TIME,AC_POWER,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,0.0,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,0.0,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,0.0,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,0.0,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,0.0,24.621525,22.165423,0.0


Save the merged file to Google Drive

In [25]:
output_path = folder / "plant1_aggregated_merged.csv"
merged.to_csv(output_path, index=False)

print("Saved file to:", output_path)

Saved file to: /content/drive/MyDrive/Project2_SolarPV/plant1_aggregated_merged.csv
